# ReviewIQ — Notebook 04: Review Intelligence Score (RIS)

**RIS** is ReviewIQ's proprietary 0–100 composite score for any product/ASIN.

### RIS Components
| Component | Weight | Description |
|-----------|--------|-------------|
| Sentiment Score | 25% | Weighted positive sentiment ratio |
| Authenticity Score | 25% | Inverse fake review probability |
| Aspect Coverage | 20% | Breadth of product qualities mentioned |
| Review Velocity Trend | 15% | Momentum of recent review activity |
| Competitive Position | 15% | Rating vs category average |

### Output
- RIS score 0–100 per ASIN
- Component breakdown
- Trend over time
- Percentile rank in category

In [ ]:
!pip install pandas numpy scikit-learn scipy spacy plotly -q
!python -m spacy download en_core_web_sm -q

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy.stats import percentileofscore
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

PROCESSED_DIR = Path("../../data/processed")
MODELS_DIR = Path("../../models")
OUTPUTS_DIR = Path("../../data/ris_outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROCESSED_DIR / "reviews_clean_all.parquet")
print(f"Loaded {len(df):,} reviews")

In [ ]:
# ── Component 1: Sentiment Score (0–100) ──────────────────────────────────────
# Uses star rating as proxy (replace with model predictions post-training)
# Bayesian average to handle ASINs with few reviews

GLOBAL_MEAN = df["star_rating"].mean()
C = 20  # prior weight (minimum reviews before trusting raw score)

def bayesian_avg(ratings_series, C=C, m=GLOBAL_MEAN):
    n = len(ratings_series)
    raw_mean = ratings_series.mean()
    return (n / (n + C)) * raw_mean + (C / (n + C)) * m

def sentiment_component(group):
    ba = bayesian_avg(group["star_rating"])
    # Normalize from [1,5] to [0,100]
    return (ba - 1) / 4 * 100

asin_sentiment = df.groupby("asin").apply(sentiment_component).rename("sentiment_score")
print(f"Sentiment scores computed for {len(asin_sentiment):,} ASINs")
print(asin_sentiment.describe())

In [ ]:
# ── Component 2: Authenticity Score (0–100) ───────────────────────────────────
# Rule-based fake review detector
# In production: replace with trained IsolationForest / XGBoost model

def compute_fake_risk_per_review(df: pd.DataFrame) -> pd.Series:
    risk = pd.Series(0.0, index=df.index)

    # Signal 1: Unverified + 5 star + very short
    if "verified_purchase" in df.columns:
        risk += (
            (~df["verified_purchase"]) &
            (df["star_rating"] == 5) &
            (df["word_count"] < 15)
        ).astype(float) * 0.35

    # Signal 2: Extremely short review (< 5 words)
    risk += (df["word_count"] < 5).astype(float) * 0.25

    # Signal 3: Duplicate review text
    dup_mask = df.duplicated(subset=["review_text"], keep=False)
    risk += dup_mask.astype(float) * 0.30

    # Signal 4: Rating extremity (only 1s and 5s for a reviewer)
    # Simplified: proxy by star_rating == 5 with no helpful votes and no images
    if "helpful_votes" in df.columns and "has_images" in df.columns:
        risk += (
            (df["star_rating"] == 5) &
            (df["helpful_votes"] == 0) &
            (~df["has_images"])
        ).astype(float) * 0.10

    return risk.clip(0, 1)


df["fake_risk"] = compute_fake_risk_per_review(df)

# Per-ASIN: average fake risk → authenticity score = (1 - avg_fake_risk) * 100
asin_authenticity = (
    df.groupby("asin")["fake_risk"]
    .mean()
    .apply(lambda x: (1 - x) * 100)
    .rename("authenticity_score")
)

print(f"Authenticity scores computed")
print(asin_authenticity.describe())

In [ ]:
# ── Component 3: Aspect Coverage Score (0–100) ────────────────────────────────
# Measures breadth of product qualities discussed
# Uses keyword matching per aspect dimension
# In production: replace with NER-based aspect extractor

ASPECT_KEYWORDS = {
    "quality": ["quality", "durable", "sturdy", "well-made", "build", "craftsmanship", "material"],
    "value": ["price", "value", "worth", "affordable", "expensive", "cheap", "cost"],
    "shipping": ["shipping", "delivery", "arrived", "fast", "slow", "packaging", "package"],
    "effectiveness": ["work", "works", "effective", "result", "results", "difference", "improvement"],
    "usability": ["easy", "use", "simple", "instructions", "setup", "interface", "intuitive"],
    "taste_smell": ["taste", "smell", "flavor", "scent", "fresh", "delicious", "odor"],  # food/beauty
    "size_fit": ["size", "fit", "large", "small", "fits", "tight", "loose", "xl"],
    "customer_service": ["service", "support", "response", "refund", "return", "helpful", "rude"],
}

N_ASPECTS = len(ASPECT_KEYWORDS)

def aspect_coverage_for_asin(review_texts: pd.Series) -> float:
    combined = " ".join(review_texts.str.lower().fillna("").tolist())
    covered = sum(
        1 for keywords in ASPECT_KEYWORDS.values()
        if any(kw in combined for kw in keywords)
    )
    return (covered / N_ASPECTS) * 100

print("Computing aspect coverage per ASIN (this takes a moment)...")
asin_aspect = (
    df.groupby("asin")["review_text"]
    .apply(aspect_coverage_for_asin)
    .rename("aspect_coverage_score")
)
print(f"Aspect coverage computed")
print(asin_aspect.describe())

In [ ]:
# ── Component 4: Review Velocity Trend Score (0–100) ──────────────────────────
# Measures recent momentum vs historical average

def velocity_trend_score(group: pd.DataFrame, recent_days=90) -> float:
    if "review_date" not in group.columns or group["review_date"].isna().all():
        return 50.0  # neutral if no date

    max_date = group["review_date"].max()
    cutoff = max_date - pd.Timedelta(days=recent_days)

    recent_count = (group["review_date"] >= cutoff).sum()
    total_count = len(group)
    days_span = max(1, (max_date - group["review_date"].min()).days)

    historical_rate = total_count / days_span  # reviews/day
    recent_rate = recent_count / recent_days

    if historical_rate == 0:
        return 50.0

    # Ratio of recent to historical (>1 = accelerating, <1 = decelerating)
    ratio = recent_rate / historical_rate
    # Convert to 0-100 (ratio=1 → 50, ratio=2 → 75, ratio=0.5 → 25)
    score = 50 * ratio
    return float(np.clip(score, 0, 100))

print("Computing velocity trends...")
if "review_date" in df.columns:
    asin_velocity = (
        df.groupby("asin")
        .apply(velocity_trend_score)
        .rename("velocity_trend_score")
    )
else:
    asin_velocity = pd.Series(50.0, index=df["asin"].unique(), name="velocity_trend_score")

print(asin_velocity.describe())

In [ ]:
# ── Component 5: Competitive Position Score (0–100) ───────────────────────────
# How does this ASIN's rating compare to its category average?

asin_category = df.groupby("asin")["category"].first()
asin_avg_rating = df.groupby("asin")["star_rating"].mean()
category_avg = df.groupby("category")["star_rating"].mean()

asin_competitive = pd.DataFrame({
    "avg_rating": asin_avg_rating,
    "category": asin_category,
})
asin_competitive["category_avg"] = asin_competitive["category"].map(category_avg)
asin_competitive["delta"] = asin_competitive["avg_rating"] - asin_competitive["category_avg"]

# Normalize delta: ±1 star = ±25 pts from 50
asin_competitive["competitive_position_score"] = (
    50 + asin_competitive["delta"] * 25
).clip(0, 100)

print("Competitive position scores computed")
print(asin_competitive["competitive_position_score"].describe())

In [ ]:
# ── Assemble RIS ──────────────────────────────────────────────────────────────
WEIGHTS = {
    "sentiment_score": 0.25,
    "authenticity_score": 0.25,
    "aspect_coverage_score": 0.20,
    "velocity_trend_score": 0.15,
    "competitive_position_score": 0.15,
}

ris_df = pd.DataFrame({
    "asin": asin_sentiment.index,
    "sentiment_score": asin_sentiment.values,
}).set_index("asin")

ris_df = ris_df.join(asin_authenticity)
ris_df = ris_df.join(asin_aspect)
ris_df = ris_df.join(asin_velocity)
ris_df = ris_df.join(asin_competitive[["competitive_position_score"]])
ris_df = ris_df.join(asin_category.rename("category"))

# Fill nulls with neutral 50
for col in WEIGHTS:
    ris_df[col] = ris_df[col].fillna(50.0)

# Compute weighted RIS
ris_df["ris_score"] = sum(
    ris_df[col] * weight for col, weight in WEIGHTS.items()
).round(1)

# Category percentile rank
ris_df["category_percentile"] = ris_df.groupby("category")["ris_score"].transform(
    lambda x: x.rank(pct=True) * 100
).round(1)

# Grade
def ris_grade(score):
    if score >= 85: return "A+"
    elif score >= 75: return "A"
    elif score >= 65: return "B"
    elif score >= 55: return "C"
    elif score >= 40: return "D"
    else: return "F"

ris_df["ris_grade"] = ris_df["ris_score"].apply(ris_grade)

print("=== RIS Score Distribution ===")
print(ris_df["ris_score"].describe().round(2))
print("\nGrade distribution:")
print(ris_df["ris_grade"].value_counts())

ris_df.reset_index().to_parquet(OUTPUTS_DIR / "ris_scores.parquet", index=False)
print(f"\n✅ RIS scores saved → {OUTPUTS_DIR / 'ris_scores.parquet'}")

In [ ]:
# ── RIS Visualization ─────────────────────────────────────────────────────────
import plotly.express as px

# Distribution by category
fig = px.box(
    ris_df.reset_index(),
    x="category",
    y="ris_score",
    color="category",
    title="RIS Score Distribution by Category",
    points="outliers",
)
fig.update_xaxes(tickangle=45)
fig.show()

# Component radar for a sample ASIN
sample_asin = ris_df.sort_values("ris_score", ascending=False).index[0]
sample = ris_df.loc[sample_asin]

radar_fig = go.Figure(data=go.Scatterpolar(
    r=[sample[col] for col in WEIGHTS],
    theta=list(WEIGHTS.keys()),
    fill="toself",
    name=sample_asin,
))
radar_fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 100])),
    title=f"RIS Component Breakdown — ASIN: {sample_asin} (RIS: {sample['ris_score']})",
)
radar_fig.show()
print(f"\nSample ASIN: {sample_asin}")
print(sample[[*WEIGHTS.keys(), "ris_score", "ris_grade", "category_percentile"]])

In [ ]:
# ── RIS API utility function (used by FastAPI backend) ────────────────────────
def get_ris_for_asin(asin: str, ris_df: pd.DataFrame) -> dict:
    """Return full RIS breakdown for a given ASIN."""
    if asin not in ris_df.index:
        return {"error": f"ASIN {asin} not found"}

    row = ris_df.loc[asin]
    return {
        "asin": asin,
        "ris_score": float(row["ris_score"]),
        "ris_grade": row["ris_grade"],
        "category": row["category"],
        "category_percentile": float(row["category_percentile"]),
        "components": {
            "sentiment_score": round(float(row["sentiment_score"]), 1),
            "authenticity_score": round(float(row["authenticity_score"]), 1),
            "aspect_coverage_score": round(float(row["aspect_coverage_score"]), 1),
            "velocity_trend_score": round(float(row["velocity_trend_score"]), 1),
            "competitive_position_score": round(float(row["competitive_position_score"]), 1),
        },
        "weights": WEIGHTS,
    }


# Test it
sample_result = get_ris_for_asin(sample_asin, ris_df)
print(json.dumps(sample_result, indent=2))